In [21]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

# Add project root to path
_REPO_ROOT = Path().resolve().parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f"Project root: {_REPO_ROOT}")

Project root: /home/lyaayladere/DeepLearningProject


# Accuracy Analysis by Dataset and Level

Analyze accuracy results from `results_llava_hf/llava-1.5-7b-hf/accuracy_question_ablation` across different datasets and levels.

In [22]:
# Load all CSV files from the accuracy_question_ablation directory
RESULTS_BASE = _REPO_ROOT / "results_llava-hf" / "llava-1.5-7b-hf" / "accuracy_question_ablation"

all_results = []

# Map dataset directories to dataset names
# Note: Only loading existential_yesno, not existential_attribute (to count only yes/no questions)
dataset_mapping = {
    "data": "vlm_levels",
    "data_v2": "vlm_levels_v2",
    "data_v3": "vlm_levels_v3",
    "existential_yesno": "vlm_levels_existential_qa_yesno",
    "existential_attribute": "vlm_levels_existential_qa_attribute",
}

for dataset_dir, dataset_name in dataset_mapping.items():
    csv_files = list((RESULTS_BASE / dataset_dir).glob("*.csv"))
    for csv_file in csv_files:
        print(f"Loading: {csv_file.name} ({dataset_name})")
        df = pd.read_csv(csv_file)
        
        # Override dataset column to match our mapping (especially for existential QA)
        df["dataset"] = dataset_name
        
        # Ensure is_correct column exists (compute if needed)
        if "is_correct" not in df.columns:
            if "prediction" in df.columns and "ground_truth" in df.columns:
                df["is_correct"] = df["prediction"] == df["ground_truth"]
            else:
                print(f"Warning: Cannot compute is_correct for {csv_file.name}")
                continue
        
        all_results.append(df)

if not all_results:
    raise ValueError(f"No CSV files found in {RESULTS_BASE}")

combined_df = pd.concat(all_results, ignore_index=True)
print(f"\nLoaded {len(combined_df)} total samples (before filtering)")

# Filter out "combined" questions if question_type column exists
if "question_type" in combined_df.columns:
    num_before = len(combined_df)
    num_combined = (combined_df["question_type"] == "combined").sum()
    # Exclude rows where question_type == "combined" (keep NaN values - those are from CSVs without question_type)
    combined_df = combined_df[combined_df["question_type"] != "combined"]
    num_after = len(combined_df)
    print(f"Excluded {num_combined} 'combined' questions")
    print(f"Remaining samples: {num_after} (after filtering)")
else:
    print("Note: No 'question_type' column found, cannot filter combined questions")

print(f"Datasets: {sorted(combined_df['dataset'].unique())}")
print(f"Levels: {sorted(combined_df['level_id'].unique())}")

Loading: visual_yesno_results.csv (vlm_levels)
Loading: visual_attribute_results.csv (vlm_levels_v2)
Loading: visual_relational_results.csv (vlm_levels_v3)
Loading: existential_yesno_results_qa_existential_yesno.csv (vlm_levels_existential_qa_yesno)
Loading: existential_attribute_results_qa_existential_attribute.csv (vlm_levels_existential_qa_attribute)

Loaded 8612 total samples (before filtering)
Excluded 716 'combined' questions
Remaining samples: 7896 (after filtering)
Datasets: ['vlm_levels', 'vlm_levels_existential_qa_attribute', 'vlm_levels_existential_qa_yesno', 'vlm_levels_v2', 'vlm_levels_v3']
Levels: ['level_0', 'level_1', 'level_2', 'level_3', 'level_4']


In [23]:
# Accuracy summary per dataset (no overall combined)
print("="*80)
print("ACCURACY ANALYSIS SUMMARY (BY DATASET)")
print("="*80)

acc_by_dataset_summary = combined_df.groupby("dataset")["is_correct"].agg(["mean", "count"]).round(4)
acc_by_dataset_summary.columns = ["accuracy", "count"]
acc_by_dataset_summary = acc_by_dataset_summary.sort_values("accuracy", ascending=False)

for dataset, row in acc_by_dataset_summary.iterrows():
    print(f"\n{dataset:<40} {row['accuracy']:.4f} ({row['accuracy']*100:.2f}%) - {int(row['count'])} samples")

ACCURACY ANALYSIS SUMMARY (BY DATASET)

vlm_levels_existential_qa_yesno          0.7662 (76.62%) - 800 samples

vlm_levels_v3                            0.6736 (67.36%) - 1676 samples

vlm_levels_v2                            0.6271 (62.71%) - 1196 samples

vlm_levels_existential_qa_attribute      0.5619 (56.19%) - 872 samples

vlm_levels                               0.5131 (51.31%) - 3352 samples


In [24]:
# Accuracy per dataset
acc_by_dataset = combined_df.groupby("dataset")["is_correct"].agg(["mean", "count"]).round(4)
acc_by_dataset.columns = ["accuracy", "count"]
acc_by_dataset = acc_by_dataset.sort_values("accuracy", ascending=False)

print("-"*80)
print("ACCURACY BY DATASET")
print("-"*80)
print(acc_by_dataset.to_string())

--------------------------------------------------------------------------------
ACCURACY BY DATASET
--------------------------------------------------------------------------------
                                     accuracy  count
dataset                                             
vlm_levels_existential_qa_yesno        0.7662    800
vlm_levels_v3                          0.6736   1676
vlm_levels_v2                          0.6271   1196
vlm_levels_existential_qa_attribute    0.5619    872
vlm_levels                             0.5131   3352


In [25]:
# Accuracy per dataset and level
acc_by_dataset_level = combined_df.groupby(["dataset", "level_id"])["is_correct"].agg(["mean", "count"]).round(4)
acc_by_dataset_level.columns = ["accuracy", "count"]
acc_by_dataset_level = acc_by_dataset_level.reset_index()

print("-"*80)
print("ACCURACY BY DATASET AND LEVEL")
print("-"*80)

# Create pivot table for dataset x level (accuracy)
pivot_acc = acc_by_dataset_level.pivot(
    index="dataset", 
    columns="level_id", 
    values="accuracy"
).fillna("N/A")

# Note: No "Total" column - datasets are kept separate
print("\nAccuracy Table:")
print(pivot_acc.to_string())

--------------------------------------------------------------------------------
ACCURACY BY DATASET AND LEVEL
--------------------------------------------------------------------------------

Accuracy Table:
level_id                             level_0  level_1  level_2  level_3  level_4
dataset                                                                         
vlm_levels                            0.5125   0.5125   0.5405   0.5185   0.5061
vlm_levels_existential_qa_attribute   0.4128   0.4643   0.6489   0.5893   0.6818
vlm_levels_existential_qa_yesno       0.8000   0.8125   0.8438   0.6812   0.6938
vlm_levels_v2                         0.5625   0.5438   0.7770   0.6567   0.5214
vlm_levels_v3                         0.5250   0.6750   0.7500   0.6761   0.6729


In [26]:
# Sample counts per dataset and level
print("-"*80)
print("SAMPLE COUNTS BY DATASET AND LEVEL")
print("-"*80)

count_pivot = acc_by_dataset_level.pivot(
    index="dataset", 
    columns="level_id", 
    values="count"
).fillna(0)
count_pivot = count_pivot.astype(int)

# Note: No "Total" column - datasets are kept separate
print(count_pivot.to_string())

--------------------------------------------------------------------------------
SAMPLE COUNTS BY DATASET AND LEVEL
--------------------------------------------------------------------------------
level_id                             level_0  level_1  level_2  level_3  level_4
dataset                                                                         
vlm_levels                               160      160      296      920     1816
vlm_levels_existential_qa_attribute      172      168      188      168      176
vlm_levels_existential_qa_yesno          160      160      160      160      160
vlm_levels_v2                            160      160      296      300      280
vlm_levels_v3                             80       80      148      460      908


In [27]:
# Alternative view: accuracy by level for each dataset (level as rows, no combined totals)
print("-"*80)
print("ACCURACY BY LEVEL FOR EACH DATASET")
print("-"*80)

acc_by_level_dataset_view = acc_by_dataset_level.pivot(
    index="level_id",
    columns="dataset",
    values="accuracy"
).fillna("N/A")

# Note: No "Total" column - datasets are kept separate
print(acc_by_level_dataset_view.to_string())

--------------------------------------------------------------------------------
ACCURACY BY LEVEL FOR EACH DATASET
--------------------------------------------------------------------------------
dataset   vlm_levels  vlm_levels_existential_qa_attribute  vlm_levels_existential_qa_yesno  vlm_levels_v2  vlm_levels_v3
level_id                                                                                                                
level_0       0.5125                               0.4128                           0.8000         0.5625         0.5250
level_1       0.5125                               0.4643                           0.8125         0.5438         0.6750
level_2       0.5405                               0.6489                           0.8438         0.7770         0.7500
level_3       0.5185                               0.5893                           0.6812         0.6567         0.6761
level_4       0.5061                               0.6818                    

In [28]:
# Accuracy by relation type AND dataset (separated by dataset)
if "relation_type" in combined_df.columns:
    acc_by_relation_dataset = combined_df.groupby(["dataset", "relation_type"])["is_correct"].agg(["mean", "count"]).round(4)
    acc_by_relation_dataset.columns = ["accuracy", "count"]
    acc_by_relation_dataset = acc_by_relation_dataset.reset_index()
    
    print("-"*80)
    print("ACCURACY BY RELATION TYPE (BY DATASET)")
    print("-"*80)
    
    # Create pivot table: relation_type x dataset (accuracy)
    pivot_rel_dataset_acc = acc_by_relation_dataset.pivot(
        index="relation_type",
        columns="dataset",
        values="accuracy"
    ).fillna("N/A")
    
    print("\nAccuracy Table:")
    print(pivot_rel_dataset_acc.to_string())
    
    # Also show counts
    print("\n" + "-"*80)
    print("SAMPLE COUNTS BY RELATION TYPE (BY DATASET)")
    print("-"*80)
    count_pivot_rel_dataset = acc_by_relation_dataset.pivot(
        index="relation_type",
        columns="dataset",
        values="count"
    ).fillna(0)
    count_pivot_rel_dataset = count_pivot_rel_dataset.astype(int)
    print(count_pivot_rel_dataset.to_string())
else:
    print("Note: 'relation_type' column not found in the data")

--------------------------------------------------------------------------------
ACCURACY BY RELATION TYPE (BY DATASET)
--------------------------------------------------------------------------------

Accuracy Table:
dataset        vlm_levels  vlm_levels_v2  vlm_levels_v3
relation_type                                          
above              0.5049         0.5959         0.8131
below              0.5194         0.6444         0.4466
left_of            0.5235         0.6171         0.6056
right_of           0.5047         0.6513         0.8263

--------------------------------------------------------------------------------
SAMPLE COUNTS BY RELATION TYPE (BY DATASET)
--------------------------------------------------------------------------------
dataset        vlm_levels  vlm_levels_v2  vlm_levels_v3
relation_type                                          
above                 824            292            412
below                 824            284            412
left_of        

In [29]:
# Accuracy by relation type, level, AND dataset (separated by dataset)
if "relation_type" in combined_df.columns:
    acc_by_relation_level_dataset = combined_df.groupby(["dataset", "relation_type", "level_id"])["is_correct"].agg(["mean", "count"]).round(4)
    acc_by_relation_level_dataset.columns = ["accuracy", "count"]
    acc_by_relation_level_dataset = acc_by_relation_level_dataset.reset_index()
    
    print("-"*80)
    print("ACCURACY BY RELATION TYPE AND LEVEL (BY DATASET)")
    print("-"*80)
    
    # Show for each dataset separately
    for dataset in sorted(combined_df["dataset"].unique()):
        dataset_data = acc_by_relation_level_dataset[acc_by_relation_level_dataset["dataset"] == dataset]
        print(f"\n{'='*80}")
        print(f"DATASET: {dataset}")
        print(f"{'='*80}")
        
        # Create pivot table for this dataset: relation_type x level
        pivot_rel_level = dataset_data.pivot(
            index="relation_type",
            columns="level_id",
            values="accuracy"
        ).fillna("N/A")
        
        print("\nAccuracy Table:")
        print(pivot_rel_level.to_string())
        
        # Also show counts
        print("\nSample Counts:")
        count_pivot = dataset_data.pivot(
            index="relation_type",
            columns="level_id",
            values="count"
        ).fillna(0).astype(int)
        print(count_pivot.to_string())
else:
    print("Note: 'relation_type' column not found in the data")

--------------------------------------------------------------------------------
ACCURACY BY RELATION TYPE AND LEVEL (BY DATASET)
--------------------------------------------------------------------------------

DATASET: vlm_levels

Accuracy Table:
level_id       level_0  level_1  level_2  level_3  level_4
relation_type                                             
above           0.5000   0.5179   0.5147   0.5043   0.5023
below           0.5000   0.5179   0.5735   0.5431   0.5000
left_of         0.5179   0.5000   0.5625   0.5175   0.5216
right_of        0.5179   0.5000   0.5125   0.5088   0.5000

Sample Counts:
level_id       level_0  level_1  level_2  level_3  level_4
relation_type                                             
above               24       56       68      232      444
below               24       56       68      232      444
left_of             56       24       80      228      464
right_of            56       24       80      228      464

DATASET: vlm_levels_existe

In [30]:
# This cell is now integrated into Cell 10 above
# (Sample counts are shown per dataset in the relation type + level analysis)

In [31]:
# Alternative view: Accuracy by level for each relation type (BY DATASET, level as rows)
if "relation_type" in combined_df.columns:
    print("-"*80)
    print("ACCURACY BY LEVEL FOR EACH RELATION TYPE (BY DATASET)")
    print("-"*80)
    
    # Show for each dataset separately
    for dataset in sorted(combined_df["dataset"].unique()):
        dataset_data = combined_df[combined_df["dataset"] == dataset]
        acc_by_level_rel = dataset_data.groupby(["level_id", "relation_type"])["is_correct"].mean().round(4).reset_index()
        
        print(f"\n{'='*80}")
        print(f"DATASET: {dataset}")
        print(f"{'='*80}")
        
        # Create pivot table: level_id x relation_type
        pivot_level_rel = acc_by_level_rel.pivot(
            index="level_id",
            columns="relation_type",
            values="is_correct"
        ).fillna("N/A")
        
        print(pivot_level_rel.to_string())

--------------------------------------------------------------------------------
ACCURACY BY LEVEL FOR EACH RELATION TYPE (BY DATASET)
--------------------------------------------------------------------------------

DATASET: vlm_levels
relation_type   above   below  left_of  right_of
level_id                                        
level_0        0.5000  0.5000   0.5179    0.5179
level_1        0.5179  0.5179   0.5000    0.5000
level_2        0.5147  0.5735   0.5625    0.5125
level_3        0.5043  0.5431   0.5175    0.5088
level_4        0.5023  0.5000   0.5216    0.5000

DATASET: vlm_levels_existential_qa_attribute
Empty DataFrame
Columns: []
Index: []

DATASET: vlm_levels_existential_qa_yesno
Empty DataFrame
Columns: []
Index: []

DATASET: vlm_levels_v2
relation_type   above   below  left_of  right_of
level_id                                        
level_0        0.5417  0.5833   0.4464    0.6786
level_1        0.6607  0.4821   0.3750    0.5833
level_2        0.7353  0.8088   0.76

In [32]:
# Separate existential yes/no and attribute datasets
existential_yesno_df = combined_df[combined_df["dataset"] == "vlm_levels_existential_qa_yesno"].copy()
existential_attribute_df = combined_df[combined_df["dataset"] == "vlm_levels_existential_qa_attribute"].copy()

print("="*80)
print("EXISTENTIAL QA DATASET SUMMARY")
print("="*80)
print(f"\nYes/No Questions: {len(existential_yesno_df)} samples")
print(f"Attribute Questions: {len(existential_attribute_df)} samples")

EXISTENTIAL QA DATASET SUMMARY

Yes/No Questions: 800 samples
Attribute Questions: 872 samples


In [33]:
# Existential Yes/No Questions - Analysis by Level
if len(existential_yesno_df) > 0:
    print("-"*80)
    print("EXISTENTIAL YES/NO QUESTIONS - ACCURACY BY LEVEL")
    print("-"*80)
    
    yesno_by_level = existential_yesno_df.groupby("level_id")["is_correct"].agg(["mean", "count"]).round(4)
    yesno_by_level.columns = ["accuracy", "count"]
    
    print("\nAccuracy by Level:")
    print(yesno_by_level.to_string())
    print(f"\nOverall Accuracy: {existential_yesno_df['is_correct'].mean():.4f} ({existential_yesno_df['is_correct'].mean()*100:.2f}%)")

--------------------------------------------------------------------------------
EXISTENTIAL YES/NO QUESTIONS - ACCURACY BY LEVEL
--------------------------------------------------------------------------------

Accuracy by Level:
          accuracy  count
level_id                 
level_0     0.8000    160
level_1     0.8125    160
level_2     0.8438    160
level_3     0.6812    160
level_4     0.6938    160

Overall Accuracy: 0.7662 (76.62%)


In [34]:
# Existential Attribute Questions - Analysis by Question Type and Level
if len(existential_attribute_df) > 0:
    print("-"*80)
    print("EXISTENTIAL ATTRIBUTE QUESTIONS - ANALYSIS BY QUESTION TYPE")
    print("-"*80)
    
    # Categorize questions by type based on question text
    def categorize_question_type(question):
        question_lower = str(question).lower()
        if "how many" in question_lower:
            return "num"
        elif "what color" in question_lower:
            return "color"
        elif "what shape" in question_lower:
            return "shape"
        else:
            return "other"
    
    existential_attribute_df = existential_attribute_df.copy()
    existential_attribute_df["question_type_cat"] = existential_attribute_df["question"].apply(categorize_question_type)
    
    # Overall by question type
    print("\nAccuracy by Question Type:")
    attr_by_type = existential_attribute_df.groupby("question_type_cat")["is_correct"].agg(["mean", "count"]).round(4)
    attr_by_type.columns = ["accuracy", "count"]
    print(attr_by_type.to_string())
    
    # Explanation for count differences
    print("\n" + "-"*80)
    print("NOTE: Question Count Distribution")
    print("-"*80)
    print("Expected: 1 num + 2 color + 2 shape = 5 questions per image")
    print("However, color questions are only generated when shapes are UNIQUE (unambiguous)")
    print(f"  - Num questions: {attr_by_type.loc['num', 'count'] if 'num' in attr_by_type.index else 0}")
    print(f"  - Color questions: {attr_by_type.loc['color', 'count'] if 'color' in attr_by_type.index else 0} (fewer than expected if some shapes are not unique)")
    print(f"  - Shape questions: {attr_by_type.loc['shape', 'count'] if 'shape' in attr_by_type.index else 0}")
    print(f"\nTotal samples: {len(existential_attribute_df)}")
    print(f"Expected if all images had 2 unique shapes: {len(existential_attribute_df) / 5 * 5} = {len(existential_attribute_df) / 5} images")
    
    # By question type AND level
    print("\n" + "-"*80)
    print("ACCURACY BY QUESTION TYPE AND LEVEL")
    print("-"*80)
    
    for qtype in ["num", "color", "shape"]:
        qtype_df = existential_attribute_df[existential_attribute_df["question_type_cat"] == qtype]
        if len(qtype_df) > 0:
            print(f"\n{qtype.upper()} Questions:")
            qtype_by_level = qtype_df.groupby("level_id")["is_correct"].agg(["mean", "count"]).round(4)
            qtype_by_level.columns = ["accuracy", "count"]
            print(qtype_by_level.to_string())

--------------------------------------------------------------------------------
EXISTENTIAL ATTRIBUTE QUESTIONS - ANALYSIS BY QUESTION TYPE
--------------------------------------------------------------------------------

Accuracy by Question Type:
                   accuracy  count
question_type_cat                 
color                0.5846    272
num                  0.3350    200
shape                0.6600    400

--------------------------------------------------------------------------------
NOTE: Question Count Distribution
--------------------------------------------------------------------------------
Expected: 1 num + 2 color + 2 shape = 5 questions per image
However, color questions are only generated when shapes are UNIQUE (unambiguous)
  - Num questions: 200
  - Color questions: 272 (fewer than expected if some shapes are not unique)
  - Shape questions: 400

Total samples: 872
Expected if all images had 2 unique shapes: 872.0 = 174.4 images

--------------------------

In [35]:
# Confusion Matrix: Existential Yes/No vs vlm_levels Yes/No
from sklearn.metrics import confusion_matrix

print("="*80)
print("CONFUSION MATRICES COMPARISON")
print("="*80)

# Get vlm_levels yes/no questions (original dataset)
vlm_levels_yesno_df = combined_df[combined_df["dataset"] == "vlm_levels"].copy()

if len(existential_yesno_df) > 0:
    print("\n" + "-"*80)
    print("EXISTENTIAL YES/NO - CONFUSION MATRIX")
    print("-"*80)
    
    cm_existential = pd.crosstab(
        existential_yesno_df["ground_truth"],
        existential_yesno_df["prediction"],
        rownames=["ground_truth"],
        colnames=["prediction"],
        dropna=False,
    ).reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    
    print(cm_existential.to_string())
    
    # Calculate metrics
    tp = cm_existential.loc["yes", "yes"] if "yes" in cm_existential.index and "yes" in cm_existential.columns else 0
    fn = cm_existential.loc["yes", "no"] if "yes" in cm_existential.index and "no" in cm_existential.columns else 0
    fp = cm_existential.loc["no", "yes"] if "no" in cm_existential.index and "yes" in cm_existential.columns else 0
    tn = cm_existential.loc["no", "no"] if "no" in cm_existential.index and "no" in cm_existential.columns else 0
    
    print(f"\nTP: {tp}, FN: {fn}, FP: {fp}, TN: {tn}")
else:
    print("No existential yes/no data available")
    cm_existential = None

if len(vlm_levels_yesno_df) > 0:
    print("\n" + "-"*80)
    print("vlm_levels YES/NO - CONFUSION MATRIX")
    print("-"*80)
    
    cm_vlm_levels = pd.crosstab(
        vlm_levels_yesno_df["ground_truth"],
        vlm_levels_yesno_df["prediction"],
        rownames=["ground_truth"],
        colnames=["prediction"],
        dropna=False,
    ).reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    
    print(cm_vlm_levels.to_string())
    
    # Calculate metrics
    tp = cm_vlm_levels.loc["yes", "yes"] if "yes" in cm_vlm_levels.index and "yes" in cm_vlm_levels.columns else 0
    fn = cm_vlm_levels.loc["yes", "no"] if "yes" in cm_vlm_levels.index and "no" in cm_vlm_levels.columns else 0
    fp = cm_vlm_levels.loc["no", "yes"] if "no" in cm_vlm_levels.index and "yes" in cm_vlm_levels.columns else 0
    tn = cm_vlm_levels.loc["no", "no"] if "no" in cm_vlm_levels.index and "no" in cm_vlm_levels.columns else 0
    
    print(f"\nTP: {tp}, FN: {fn}, FP: {fp}, TN: {tn}")
else:
    print("No vlm_levels yes/no data available")
    cm_vlm_levels = None

CONFUSION MATRICES COMPARISON

--------------------------------------------------------------------------------
EXISTENTIAL YES/NO - CONFUSION MATRIX
--------------------------------------------------------------------------------
prediction    yes   no
ground_truth          
yes           391    9
no            178  222

TP: 391, FN: 9, FP: 178, TN: 222

--------------------------------------------------------------------------------
vlm_levels YES/NO - CONFUSION MATRIX
--------------------------------------------------------------------------------
prediction     yes  no
ground_truth          
yes           1662  14
no            1618  58

TP: 1662, FN: 14, FP: 1618, TN: 58


In [36]:
# -----------------------------------------------------------------------------
# Black vs White background analysis (BY DATASET, and BY DATASET x LEVEL)
# Background is inferred from image_id suffix: *_b (black) or *_w (white)
# -----------------------------------------------------------------------------
print("=" * 80)
print("BLACK vs WHITE BACKGROUND ACCURACY (BY DATASET)")
print("=" * 80)

if "image_id" not in combined_df.columns:
    print("No 'image_id' column found in combined_df; cannot compute black/white split.")
else:
    bw_df = combined_df.copy()

    # Infer background from suffix
    bw_df["background"] = bw_df["image_id"].astype(str).str.extract(r"_(b|w)$")[0]
    bw_df["background"] = bw_df["background"].map({"b": "black", "w": "white"})

    num_unknown = int(bw_df["background"].isna().sum())
    if num_unknown:
        print(
            f"Warning: {num_unknown} rows have unknown background (image_id not ending with _b/_w). "
            "They will be excluded."
        )

    bw_df = bw_df.dropna(subset=["background"]).copy()

    # Overall: dataset x background
    acc_by_dataset_bg = (
        bw_df.groupby(["dataset", "background"])["is_correct"]
        .agg(["mean", "count"])
        .round(4)
        .reset_index()
    )
    acc_by_dataset_bg.columns = ["dataset", "background", "accuracy", "count"]

    print("\n" + "-" * 80)
    print("ACCURACY BY DATASET x BACKGROUND")
    print("-" * 80)
    print(acc_by_dataset_bg.sort_values(["dataset", "background"]).to_string(index=False))

    # Pivot: per dataset (black/white columns)
    pivot_acc_bg = acc_by_dataset_bg.pivot(index="dataset", columns="background", values="accuracy")
    pivot_n_bg = acc_by_dataset_bg.pivot(index="dataset", columns="background", values="count")

    print("\n" + "-" * 80)
    print("ACCURACY PIVOT (rows=dataset, cols=background)")
    print("-" * 80)
    print(pivot_acc_bg.to_string())

    print("\n" + "-" * 80)
    print("COUNTS PIVOT (rows=dataset, cols=background)")
    print("-" * 80)
    print(pivot_n_bg.to_string())

    # By dataset x level x background
    print("\n" + "=" * 80)
    print("BLACK vs WHITE BACKGROUND ACCURACY (BY DATASET x LEVEL)")
    print("=" * 80)

    acc_by_dataset_level_bg = (
        bw_df.groupby(["dataset", "level_id", "background"])["is_correct"]
        .agg(["mean", "count"])
        .round(4)
        .reset_index()
    )
    acc_by_dataset_level_bg.columns = ["dataset", "level_id", "background", "accuracy", "count"]

    for dataset in sorted(acc_by_dataset_level_bg["dataset"].unique()):
        sub = acc_by_dataset_level_bg[acc_by_dataset_level_bg["dataset"] == dataset]

        acc_p = sub.pivot(index="level_id", columns="background", values="accuracy")
        n_p = sub.pivot(index="level_id", columns="background", values="count")

        print("\n" + "-" * 80)
        print(f"DATASET: {dataset}")
        print("-" * 80)
        print("Accuracy (rows=level_id, cols=background):")
        print(acc_p.to_string())
        print("\nCounts (rows=level_id, cols=background):")
        print(n_p.to_string())


BLACK vs WHITE BACKGROUND ACCURACY (BY DATASET)

--------------------------------------------------------------------------------
ACCURACY BY DATASET x BACKGROUND
--------------------------------------------------------------------------------
                            dataset background  accuracy  count
                         vlm_levels      black    0.5167   1676
                         vlm_levels      white    0.5095   1676
vlm_levels_existential_qa_attribute      black    0.5711    436
vlm_levels_existential_qa_attribute      white    0.5528    436
    vlm_levels_existential_qa_yesno      black    0.7625    400
    vlm_levels_existential_qa_yesno      white    0.7700    400
                      vlm_levels_v2      black    0.6154    598
                      vlm_levels_v2      white    0.6388    598
                      vlm_levels_v3      black    0.6695    838
                      vlm_levels_v3      white    0.6778    838

---------------------------------------------------

In [37]:
# ============================================================================
# Unmasked vs Always-Masked vs Background-Masked Accuracy
# - Loads three result roots:
#   * accuracy_question_ablation              -> condition = 'unmasked'
#   * always_masked_accuracy_ablation        -> condition = 'always_masked'
#   * background_masked_accuracy_ablation    -> condition = 'background_masked'
# - Compares accuracy BY DATASET and BY DATASET x LEVEL
# ============================================================================
from pathlib import Path

print("=" * 80)
print("UNMASKED vs MASKED COMPARISON (BY DATASET and BY DATASET x LEVEL)")
print("=" * 80)

BASE_ROOT = _REPO_ROOT / "results_llava-hf" / "llava-1.5-7b-hf"

condition_roots = {
    "unmasked": BASE_ROOT / "accuracy_question_ablation",
    "always_masked": BASE_ROOT / "always_masked_accuracy_ablation",
    "background_masked": BASE_ROOT / "background_masked_accuracy_ablation",
}

# Same mapping as in the main load cell
cond_dataset_mapping = {
    "data": "vlm_levels",
    "data_v2": "vlm_levels_v2",
    "data_v3": "vlm_levels_v3",
    "existential_yesno": "vlm_levels_existential_qa_yesno",
    "existential_attribute": "vlm_levels_existential_qa_attribute",
}

all_cond_results = []

for cond, root in condition_roots.items():
    if not root.exists():
        print(f"Warning: condition '{cond}' root does not exist: {root}")
        continue

    print("\n" + "-" * 80)
    print(f"Loading condition: {cond} from {root}")
    print("-" * 80)

    for dataset_dir, dataset_name in cond_dataset_mapping.items():
        subdir = root / dataset_dir
        if not subdir.exists():
            continue

        csv_files = sorted(subdir.glob("*.csv"))
        if not csv_files:
            continue

        for csv_file in csv_files:
            print(f"  {cond}: {dataset_dir} -> {csv_file.name}")
            df = pd.read_csv(csv_file)

            # Normalize columns
            df["dataset"] = dataset_name
            if "level_id" not in df.columns:
                # If level information is missing, skip from level-based analyses
                df["level_id"] = "unknown"

            if "is_correct" not in df.columns:
                if "prediction" in df.columns and "ground_truth" in df.columns:
                    df["is_correct"] = df["prediction"] == df["ground_truth"]
                else:
                    print(f"    Warning: cannot compute is_correct for {csv_file.name} in {cond}")
                    continue

            # Apply same filtering rule: drop question_type == 'combined' if present
            if "question_type" in df.columns:
                df = df[df["question_type"] != "combined"].copy()

            df["condition"] = cond
            all_cond_results.append(df)

if not all_cond_results:
    print("No results loaded for any condition; nothing to compare.")
else:
    comp_df = pd.concat(all_cond_results, ignore_index=True)

    print("\n" + "=" * 80)
    print("ACCURACY BY CONDITION x DATASET")
    print("=" * 80)

    acc_by_cond_dataset = (
        comp_df.groupby(["condition", "dataset"])["is_correct"]
        .agg(["mean", "count"])
        .round(4)
        .reset_index()
    )
    acc_by_cond_dataset.columns = ["condition", "dataset", "accuracy", "count"]

    # Pretty print grouped by condition
    for cond in sorted(acc_by_cond_dataset["condition"].unique()):
        sub = acc_by_cond_dataset[acc_by_cond_dataset["condition"] == cond]
        print("\n" + "-" * 80)
        print(f"CONDITION: {cond} - ACCURACY BY DATASET")
        print("-" * 80)
        print(sub.sort_values("dataset").to_string(index=False))

    # Pivot view: rows = dataset, cols = (condition), values = accuracy
    pivot_acc_ds = acc_by_cond_dataset.pivot(index="dataset", columns="condition", values="accuracy")
    pivot_n_ds = acc_by_cond_dataset.pivot(index="dataset", columns="condition", values="count")

    print("\n" + "-" * 80)
    print("PIVOT: ACCURACY (rows=dataset, cols=condition)")
    print("-" * 80)
    print(pivot_acc_ds.to_string())

    print("\n" + "-" * 80)
    print("PIVOT: COUNTS (rows=dataset, cols=condition)")
    print("-" * 80)
    print(pivot_n_ds.to_string())

    # ------------------------------------------------------------------
    # BY CONDITION x DATASET x LEVEL
    # ------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("ACCURACY BY CONDITION x DATASET x LEVEL")
    print("=" * 80)

    acc_by_cond_ds_level = (
        comp_df.groupby(["condition", "dataset", "level_id"])["is_correct"]
        .agg(["mean", "count"])
        .round(4)
        .reset_index()
    )
    acc_by_cond_ds_level.columns = ["condition", "dataset", "level_id", "accuracy", "count"]

    for cond in sorted(acc_by_cond_ds_level["condition"].unique()):
        print("\n" + "-" * 80)
        print(f"CONDITION: {cond} - ACCURACY BY DATASET x LEVEL")
        print("-" * 80)

        sub_cond = acc_by_cond_ds_level[acc_by_cond_ds_level["condition"] == cond]

        for dataset in sorted(sub_cond["dataset"].unique()):
            sub = sub_cond[sub_cond["dataset"] == dataset]

            acc_p = sub.pivot(index="level_id", columns="condition", values="accuracy")
            n_p = sub.pivot(index="level_id", columns="condition", values="count")

            print("\n" + "-" * 40)
            print(f"DATASET: {dataset}")
            print("-" * 40)
            print("Accuracy (rows=level_id, cols=condition):")
            print(acc_p.to_string())
            print("\nCounts (rows=level_id, cols=condition):")
            print(n_p.to_string())

UNMASKED vs MASKED COMPARISON (BY DATASET and BY DATASET x LEVEL)

--------------------------------------------------------------------------------
Loading condition: unmasked from /home/lyaayladere/DeepLearningProject/results_llava-hf/llava-1.5-7b-hf/accuracy_question_ablation
--------------------------------------------------------------------------------
  unmasked: data -> visual_yesno_results.csv
  unmasked: data_v2 -> visual_attribute_results.csv
  unmasked: data_v3 -> visual_relational_results.csv
  unmasked: existential_yesno -> existential_yesno_results_qa_existential_yesno.csv
  unmasked: existential_attribute -> existential_attribute_results_qa_existential_attribute.csv

--------------------------------------------------------------------------------
Loading condition: always_masked from /home/lyaayladere/DeepLearningProject/results_llava-hf/llava-1.5-7b-hf/always_masked_accuracy_ablation
--------------------------------------------------------------------------------
  alwa

In [38]:
# Use comp_df from previous cell if it exists, otherwise recreate it
if 'comp_df' not in locals():
    print("Warning: comp_df not found. Recreating from condition roots...")
    BASE_ROOT = _REPO_ROOT / "results_llava-hf" / "llava-1.5-7b-hf"

In [39]:
# Use comp_df from previous cell if it exists, otherwise recreate it
if 'comp_df' not in locals():
    print("Warning: comp_df not found. Recreating from condition roots...")
    BASE_ROOT = _REPO_ROOT / "results_llava-hf" / "llava-1.5-7b-hf"

In [40]:
# Ensure comp_df exists (from comparison cell)
if 'comp_df' not in locals() or comp_df is None or len(comp_df) == 0:
    print("Error: comp_df not found/empty. Run the comparison cell first.")
else:
    df = comp_df[comp_df["dataset"] == "vlm_levels"].copy()

    if len(df) == 0:
        print("No vlm_levels rows in comp_df.")
    elif "image_id" not in df.columns:
        print("No image_id column found; cannot split by black/white.")